In [1]:
from pathlib import Path
from collections import Counter
import math
import re
import importlib
import torch
from torch.utils.data import DataLoader
from tokenizers import Tokenizer
import scratch_qwen_adaptive
importlib.reload(scratch_qwen_adaptive)  # ensure notebook uses the current model/training implementation
from scratch_qwen_adaptive import (
    AdaptiveMathTransformer, MathSolutionDataset, ModelConfig,
    collate_math, evaluate, flash_attention_status, generate_solution,
    grouped_split, load_big_math_jsonl, load_math_parquets, make_loaders,
    make_probe_loader, seed_everything, template_key, train_model, train_tokenizer,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print(flash_attention_status(device))

Device: cuda
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU. Attention calls use PyTorch SDPA, which dispatches to Flash Attention when the device, dtype, and mask are supported.


In [2]:
# Load 520K Big Math examples and Hendrycks MATH. Reserve the remaining test rows.
train_questions, train_answers, test_frame = load_math_parquets('hendrycks_math')

# Move 4,000 of the 5,000 MATH test problems into training (+53% data); keep 1,000 as the final test set.
import pandas as pd
_p = torch.randperm(len(test_frame), generator=torch.Generator().manual_seed(7)).tolist()
extra, test_frame = test_frame.iloc[_p[:4000]], test_frame.iloc[_p[4000:]]
train_questions = pd.concat([train_questions, extra.iloc[:, 0].fillna('').astype(str)], ignore_index=True)
train_answers = pd.concat([train_answers, extra.iloc[:, -1].fillna('').astype(str)], ignore_index=True)
big_questions, big_responses, big_answers, big_chapters, big_ids = load_big_math_jsonl(
    'big_math/datasetv2.jsonl/dataset.jsonl'
)
all_questions = big_questions + train_questions.tolist()
all_responses = big_responses + train_answers.tolist()
all_groups = big_chapters + ['hendrycks_math'] * len(train_questions)
# One split group per Big Math template; Hendrycks rows are individually distinct.
all_templates = [template_key(pid) for pid in big_ids] + [f'hendrycks_{i}' for i in range(len(train_questions))]
print('Big Math templates:', len(set(all_templates[:len(big_ids)])))
print('Big Math examples:', len(big_questions))
print('Combined training examples:', len(all_questions))
print('Test examples:', 0 if test_frame is None else len(test_frame))
print('Question sample:', all_questions[0][:250])
print('Solution sample:', all_responses[0][:250])

# Hold out WHOLE chapter families, not random rows. A random row split leaves every
# validation prompt on a template that training already saw, which is why the old
# validation loss (0.19) was optimistic and the test loss (2.00) was not.
# Swap all_groups for all_templates to hold out individual templates instead.
VALID_CHAPTERS = ('olympiad_geometry', 'olympiad_trigonometry',
                  'olympiad_combinatorics', 'olympiad_algebra')
fit_idx, valid_idx, valid_chapters = grouped_split(
    all_groups, valid_groups=VALID_CHAPTERS, purge=all_templates
)
fit_group_counts = Counter(all_groups[i] for i in fit_idx)
print(f'Fit: {len(fit_idx):,} | validation: {len(valid_idx):,} ({len(valid_idx)/len(all_questions):.2%})')
print('Held-out chapters:', valid_chapters)
print(f'Held-out templates: {len({all_templates[i] for i in valid_idx})}')
print('Validation chapters:', Counter(all_groups[i] for i in valid_idx))
fit_templates = {all_templates[i] for i in fit_idx}
leaked = sum(all_templates[i] in fit_templates for i in valid_idx)
print(f'Validation prompts whose template appears in training: {leaked}/{len(valid_idx)}')
print('Training examples per chapter/group:', fit_group_counts)

Big Math templates: 190
Big Math examples: 520416
Combined training examples: 531916
Test examples: 1000
Question sample: Find 1 + 2 + ... + 97. What is its exact value?
Solution sample: Goal: compute value for template NN_001. Givens: n=97 Using expression: (n*(n+1))//2 Substitute values: (97*(97+1))//2 Compute result → 4753. Therefore the final answer is 4753.
Fit: 503,919 | validation: 27,997 (5.26%)
Held-out chapters: ['olympiad_geometry', 'olympiad_trigonometry', 'olympiad_combinatorics', 'olympiad_algebra']
Held-out templates: 81
Validation chapters: Counter({'olympiad_algebra': 14680, 'olympiad_trigonometry': 5019, 'olympiad_geometry': 4404, 'olympiad_combinatorics': 3894})
Validation prompts whose template appears in training: 0/27997
Training examples per chapter/group: Counter({'quadratics': 115629, 'algebraic_expressions': 83581, 'linear_equations': 68359, 'factors_multiples': 59861, 'modular_arithmetic': 53213, 'integers': 40699, 'natural_numbers': 37511, 'olympiad_number_t

In [3]:
# Train a byte-level BPE from scratch on questions and worked solutions.
VOCAB_SIZE = 8000
BLOCK_SIZE = 512
BATCH_SIZE = 32
tokenizer_path = Path('scratch_math_tokenizer.json')
tokenizer = train_tokenizer(
    (all_questions[i] for i in fit_idx),
    (all_responses[i] for i in fit_idx),
    vocab_size=VOCAB_SIZE
)
tokenizer.save(str(tokenizer_path))

# Square-root inverse-frequency sampling exposes rare chapters more often
# without giving every tiny category the same mass as quadratics.
train_weights = [1.0 / math.sqrt(fit_group_counts[all_groups[i]]) for i in fit_idx]
train_loader, valid_loader = make_loaders(
    all_questions, all_responses, all_questions, all_responses, tokenizer,
    block_size=BLOCK_SIZE, batch_size=BATCH_SIZE, num_workers=0,
    train_indices=fit_idx, valid_indices=valid_idx, train_weights=train_weights
)
# Same metric as validation (answer tokens only, dropout off, EMA weights) on a
# fixed slice of TRAINING data, so the two losses are actually comparable.
probe_loader = make_probe_loader(
    all_questions, all_responses, tokenizer, fit_idx,
    block_size=BLOCK_SIZE, batch_size=BATCH_SIZE, max_examples=2048
)
print('Vocabulary:', tokenizer.get_vocab_size())
print('Train batches:', len(train_loader), '| validation batches:', len(valid_loader),
      '| probe batches:', len(probe_loader))
# Confirm answer tokens are supervised and prompt/padding positions are ignored.
example_ids, example_labels, example_mask = next(iter(train_loader))
print('Batch shapes:', example_ids.shape, example_labels.shape, example_mask.shape)
print('Supervised answer tokens:', (example_labels[:, 1:] != -100).sum().item())

Vocabulary: 8000
Train batches: 15748 | validation batches: 875 | probe batches: 64
Batch shapes: torch.Size([32, 212]) torch.Size([32, 212]) torch.Size([32, 212])
Supervised answer tokens: 2968


In [4]:
# Compact scratch model; increase width/depth if GPU memory and dataset size permit.
config = ModelConfig(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=192,
    num_heads=8,
    num_kv_heads=2,  # grouped-query attention
    num_layers=6,
    max_seq_len=BLOCK_SIZE,
    exit_layers=(2, 4),
    dropout=0.1,          # modest regularization; Big Math greatly expands the corpus
    tie_embeddings=True,  # share input/output embeddings
)
model = AdaptiveMathTransformer(config)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 5,632,128


In [5]:
best_val_loss = train_model(
    model, train_loader, valid_loader, device,
    epochs=8, learning_rate=3e-4, weight_decay=0.1, warmup_steps=500, patience=3,
    checkpoint='big_math_adaptive_scratch_best.pt', probe_loader=probe_loader
)
print('Best validation token loss:', best_val_loss)

epoch 001 | train/token 1.1716 | train-probe/answer 0.2497 | val/unseen-answer 4.8491 | lr 2.90e-04
epoch 002 | train/token 0.7911 | train-probe/answer 0.2321 | val/unseen-answer 4.8419 | lr 2.61e-04
epoch 003 | train/token 0.7580 | train-probe/answer 0.2201 | val/unseen-answer 4.7747 | lr 2.18e-04
epoch 004 | train/token 0.7345 | train-probe/answer 0.2129 | val/unseen-answer 4.8336 | lr 1.66e-04
epoch 005 | train/token 0.7197 | train-probe/answer 0.2070 | val/unseen-answer 4.8299 | lr 1.14e-04
epoch 006 | train/token 0.7082 | train-probe/answer 0.2054 | val/unseen-answer 4.7820 | lr 6.98e-05
Early stopping: validation loss did not improve for 3 epochs.
Best validation token loss: 4.774674624857021


In [6]:
# Load the best validation checkpoint (still entirely trained from scratch).
checkpoint = torch.load('big_math_adaptive_scratch_best.pt', map_location=device)
model.load_state_dict(checkpoint['model'])
model.to(device).eval()
print('Best epoch:', checkpoint['epoch'], '| validation token loss:', checkpoint['best_val_loss'])

# Optional held-out answer-token loss; the test set is not used for training.
if test_frame is not None and len(test_frame):
    test_questions = test_frame.iloc[:, 0].fillna('').astype(str).tolist()
    test_answers = test_frame.iloc[:, -1].fillna('').astype(str).tolist()
    test_ds = MathSolutionDataset(test_questions, test_answers, tokenizer, block_size=BLOCK_SIZE)
    pad_id = tokenizer.token_to_id('[PAD]')
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                             collate_fn=lambda rows: collate_math(rows, pad_id))
    print('Hendrycks test answer-token loss:', evaluate(model, test_loader, device))
    print('Compare with val/unseen-answer above: the remainder is format/style shift.')

# Quick exact-answer check on 16 validation questions from unseen templates.
big_valid_indices = [i for i in valid_idx if i < len(big_answers)][:16]
for adaptive in (False, True):
    correct, depths = 0, []
    for i in big_valid_indices:
        prediction, used = generate_solution(
            model, tokenizer, all_questions[i], device, max_new_tokens=96,
            use_adaptive=adaptive, confidence_threshold=0.95,
            adaptive_min_depth=2, return_depths=True
        )
        numbers = re.findall(r'(?<![A-Za-z])\d[\d,]*', prediction)
        correct += bool(numbers and numbers[-1].replace(',', '') == str(big_answers[i]))
        depths.extend(used)
    mode = 'adaptive' if adaptive else 'full-depth'
    print(f'{mode} Big Math exact: {correct}/{len(big_valid_indices)}; mean layers/token: {sum(depths)/max(len(depths),1):.2f}')

Best epoch: 3 | validation token loss: 4.774674624857021
Hendrycks test answer-token loss: 2.2374525820004836
Compare with val/unseen-answer above: the remainder is format/style shift.
full-depth Big Math exact: 1/16; mean layers/token: 6.00
adaptive Big Math exact: 1/16; mean layers/token: 5.54


In [7]:
question = 'Solve: If 2x +4= 4, find x.'
# Compare full-depth and adaptive decoding on the same example.
answer = generate_solution(model, tokenizer, question, device,
                          max_new_tokens=128, use_adaptive=False)
adaptive_answer, layers_used = generate_solution(model, tokenizer, question, device,
                                   max_new_tokens=128, use_adaptive=True,
                                   confidence_threshold=0.95, adaptive_min_depth=2,
                                   return_depths=True)
print('Full depth:', answer)
print('Adaptive depth:', adaptive_answer)
print('Layers used per generated token:', layers_used)

Full depth: Goal: compute value for template LE_001.
Givens: a=2, b=4, c=4
Using expression: (c - b) / a
Substitute values: (4 - 4) / 2
Compute result → 0.0. Therefore the final answer is 0. 0.
Adaptive depth: Goal: compute value for template LE_001.
Givens: a=2, b=4, c=4
Using expression: (c - b) / a
Substitute values: (4 - 4) / 2
Compute result → 0.0. Therefore the final answer is 0. 0.
Layers used per generated token: [4, 2, 2, 2, 2, 2, 6, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 6, 2, 6, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [8]:
import os
print(os.path.exists('big_math_adaptive_scratch_best.pt'))

True
